# Notebook 11 — Executive Summary

**Project:** Starbucks Customer Voice Intelligence: A U.S. Coffee Chain Market Study  
**Dataset:** Yelp Open Dataset · U.S. reviews 2017–2021  
**Scope:** 11,675 Starbucks reviews · 6,866 Dunkin' reviews · 362,334 independent café reviews

## 0. Environment setup

In [1]:
import sys
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

INTERIM_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Environment ready.")

Environment ready.


## 1. Key metrics at a glance

In [2]:
df = pd.read_parquet(INTERIM_DIR / "analysis_ready.parquet")

scored_brands = ["Starbucks", "Dunkin'", "Independent Café"]
rows = []
for b in scored_brands:
    g = df[df["brand_category"] == b]
    elite = g[g["user_activity_tier"] == "Elite"]["review_stars"].mean()
    rows.append({
        "Brand":           b,
        "Reviews":         f"{len(g):,}",
        "Avg Stars":       f"{g['review_stars'].mean():.2f}",
        "% Positive":      f"{(g['star_tier']=='Positive').mean()*100:.1f}%",
        "% Critical":      f"{(g['star_tier']=='Critical').mean()*100:.1f}%",
        "Avg Sentiment":   f"{g['sentiment_score'].mean():.3f}",
        "Elite Avg Stars": f"{elite:.2f}",
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

           Brand Reviews Avg Stars % Positive % Critical Avg Sentiment Elite Avg Stars
       Starbucks  11,675      2.90      42.4%      47.6%         0.277            3.60
         Dunkin'   6,866      2.05      21.5%      71.2%        -0.020            3.01
Independent Café 362,334      4.01      73.7%      17.2%         0.706            4.10


## 2. Brand scorecard — topic positive rate

In [3]:
topic_dims = ["Service", "Food Quality", "Ambiance", "Wait Time", "Cleanliness"]

scores = {}
for b in scored_brands:
    g = df[df["brand_category"] == b]
    rates = {}
    for t in topic_dims:
        mask  = g["topic_tag"] == t
        total = mask.sum()
        pos   = (mask & (g["star_tier"] == "Positive")).sum()
        rates[t] = round(pos / total * 100, 1) if total > 0 else 0
    scores[b] = rates

theta = topic_dims + [topic_dims[0]]
brand_styles = {
    "Starbucks":        {"color": "#00704A", "dash": "solid"},
    "Dunkin'":          {"color": "#d62728", "dash": "dash"},
    "Independent Café": {"color": "#aaaaaa", "dash": "dot"},
}

fig = go.Figure()
for brand, style in brand_styles.items():
    vals = [scores[brand][d] for d in topic_dims] + [scores[brand][topic_dims[0]]]
    fig.add_trace(go.Scatterpolar(
        r=vals, theta=theta, name=brand,
        mode="lines+markers",
        line=dict(color=style["color"], dash=style["dash"], width=2.5),
        marker=dict(color=style["color"], size=8),
        fill="toself", fillcolor=style["color"], opacity=0.12,
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True, range=[0, 100],
            tickvals=[0, 25, 50, 75, 100], ticksuffix="%",
            gridcolor="#dddddd", linecolor="#dddddd",
        ),
        angularaxis=dict(gridcolor="#dddddd", linecolor="#dddddd"),
    ),
    title=dict(text="Brand Scorecard — Topic Positive Rate (%)", font=dict(size=16), x=0.5),
    legend=dict(orientation="h", yanchor="bottom", y=-0.18, xanchor="center", x=0.5),
    paper_bgcolor="white",
    width=620, height=580,
    margin=dict(t=80, b=90, l=60, r=60),
)
fig.write_html(str(FIGURES_DIR / "11_scorecard_summary.html"))
fig.show()

## 3. Five key findings

**1. Starbucks significantly underperforms the market — but outperforms its closest chain competitor.**
Starbucks averages 2.90 stars against an independent café market benchmark of 3.97 — a 1.07-star gap. However, Dunkin' performs even worse at 2.05 stars with a 71.2% critical review rate. The performance deficit is a chain-format problem, not a Starbucks-specific one. Both brands fall well below independent cafés across every operational dimension.

**2. Service is both the brand's greatest strength and its most volatile risk.**
Service accounts for 80.7% of positive review volume — when Starbucks earns praise, it almost always comes from a person, not a product. But service also drives 76.7% of critical review volume, with a 46.5% failure rate. A 43.6% positive rate against a 46.5% failure rate means front-line execution is effectively a coin flip, and this single variable determines the majority of customer outcomes.

**3. Ambiance is the brand's only reliable advantage over Dunkin'.**
Starbucks scores 58.2% positive rate on Ambiance — its strongest topic and the dimension where the gap versus Dunkin' (41.7%) is clearest. In-store environment is what differentiates Starbucks from a purely transactional drive-through brand. Cleanliness, at a 76.7% failure rate, directly undermines this advantage.

**4. Negative signal is concentrated among first-time and casual reviewers.**
Casual reviewers (fewer than 10 lifetime Yelp reviews) account for 64.3% critical reviews and average 2.43 stars. Elite reviewers average 3.60 stars with only 19.6% critical. The 1.17-star gap between these groups is larger than the gap between Starbucks and the overall market benchmark. New and infrequent customers are consistently having the worst experience — a meaningful risk for acquisition and retention.

**5. Weekend demand exposes operational limits.**
Saturday and Sunday generate the highest review volumes of the week while producing the lowest ratings: 2.71 stars versus 2.99 on weekdays. The weekend critical rate is 53.2% against 45.0% on weekdays. September is the weakest month of the year at 2.78 stars, coinciding with back-to-school traffic surges and seasonal menu transitions.

## 4. Strategic recommendations

**R1 — Address weekend staffing and queue management first.**
The weekend rating gap is the most operationally tractable finding in this dataset. A 0.28-star improvement on weekends alone — if sustained at current weekday levels — would lift the overall average by approximately 0.09 stars. Targeted staffing ratios, pre-shift preparation standards, and drive-through protocols on high-volume days represent the highest-ROI intervention available.

**R2 — Treat cleanliness as a threshold requirement, not a differentiator.**
With a 76.7% failure rate, cleanliness generates almost no positive signal and substantial negative signal. No amount of service improvement fully compensates for hygiene complaints in a café environment. Facility audit schedules and visible cleanliness standards during peak hours should be treated as non-negotiable baseline requirements.

**R3 — Design the first-visit experience specifically for casual customers.**
Casual reviewers — who are likely also first-time or infrequent visitors — experience Starbucks at 2.43 stars on average. This is the group most likely to form lasting brand impressions and most likely to be price-sensitive. Onboarding touchpoints (app ordering guidance, loyalty program entry, staff acknowledgment of new visitors) could materially improve first-impression outcomes.

**R4 — Use ambiance as the primary brand differentiator versus Dunkin'.**
Ambiance is the one dimension where Starbucks clearly outperforms Dunkin' and approaches the lower range of the independent café benchmark. Investments in seating quality, noise management, and visual environment reinforce the positioning gap between a premium café experience and a transactional drive-through. This dimension has the highest positive rate (58.2%) and the most room to convert neutral customers into advocates.

---

## 5. Analytical limitations

- **Yelp review bias:** Yelp reviewers skew toward customers with strong opinions — highly satisfied or highly dissatisfied. Silent majority experience is not captured.
- **Date range:** Dataset covers 2017–2021. COVID-19 caused a 40% review volume decline in 2020 that partially recovered in 2021. Post-pandemic behavior patterns may differ.
- **Topic tagging:** Topic classification uses rule-based keyword matching. A single topic is assigned per review; multi-topic reviews are simplified.
- **Geographic scope:** 13 U.S. states and over 500 cities represented in the Yelp dataset. Results may not generalize to all U.S. markets or international operations.
- **Dunkin' sample:** 6,866 Dunkin' reviews are sufficient for directional comparison but are drawn from the same geographic subset and may not represent Dunkin's full U.S. footprint.